# 6. SiO₂ zero-field MD: dielectric function, IR, and Raman

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/06_sio2_md_spectroscopy.ipynb)

This notebook runs fixed-cell, zero-field NVT molecular dynamics with a bundled α-quartz MACEField model. It records polarization and the electronic susceptibility tensor, computes their autocorrelation functions, and makes exploratory frequency-dependent dielectric, IR, and Raman plots.

The dielectric function uses the classical polarization fluctuation relation. IR intensity is based on the polarization spectrum; the Raman plot is a relative tensor-autocorrelation proxy. A short trajectory is a workflow demonstration, not a converged spectrum. Use longer trajectories, block/error analysis, and convergence checks for scientific reporting.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## Run zero-field molecular dynamics

The default trajectory is intentionally short for Colab. With a 0.5 fs step, 1000 steps give only 0.5 ps and a coarse frequency grid. Increase MACEFIELD_SIO2_MD_STEPS after confirming the model, device, and memory budget.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from ase import units
from ase.io import read
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.optimize import BFGS
from mace.calculators import MACECalculator
import torch

MODEL = DATA_DIR / "sio2/MACE-field-SiO2.model"
SIO2_DATA = DATA_DIR / "sio2/SiO2-preprocessed.extxyz"
atoms = read(SIO2_DATA, index=0)
atoms.info.clear()
for key in list(atoms.arrays):
    if key not in {"numbers", "positions"}:
        atoms.arrays.pop(key)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
relax_calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=DEVICE,
    default_dtype="float32",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=False,
    compute_becs=False,
    compute_polarizability=False,
)
atoms.calc = relax_calc
BFGS(atoms, logfile=None).run(fmax=0.05, steps=40)

temperature_K = 300.0
dt_fs = 0.5
default_steps = 1000 if DEVICE == "cuda" else 250
n_steps = int(os.environ.get("MACEFIELD_SIO2_MD_STEPS", default_steps))
rng = np.random.default_rng(31)
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K, rng=rng)
Stationary(atoms)

calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=DEVICE,
    default_dtype="float32",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=False,
    compute_polarizability=True,
)
atoms.calc = calc
dyn = Langevin(
    atoms,
    timestep=dt_fs * units.fs,
    temperature_K=temperature_K,
    friction=0.02 / units.fs,
    rng=rng,
)

polarization_series = []
susceptibility_series = []

def sample_response():
    atoms.get_potential_energy()
    polarization_series.append(
        np.asarray(calc.results["polarization"], dtype=float).copy()
    )
    susceptibility_series.append(
        np.asarray(calc.results["polarizability"], dtype=float).reshape(3, 3).copy()
    )

sample_response()
dyn.attach(sample_response, interval=1)
dyn.run(n_steps)

polarization_series = np.asarray(polarization_series)
susceptibility_series = np.asarray(susceptibility_series)
time_fs = np.arange(len(polarization_series)) * dt_fs
volume_A3 = atoms.get_volume()
print("device:", DEVICE)
print("temperature (K):", temperature_K)
print("frames:", len(time_fs), "| duration (ps):", time_fs[-1] / 1000)
print("frequency spacing (cm⁻¹):",
      1.0 / (len(time_fs) * dt_fs * 1.0e-15) / (299792458.0 * 100.0))


## Build polarization and polarizability autocorrelation functions

Polarization is recorded in e/Å². The calculator's volume-normalized susceptibility is dimensionless and proportional to the extensive cell polarizability at this fixed volume. The spectra below are normalized for shape, not absolute scattering cross sections.


In [ ]:
def autocorrelation_matrix(series):
    values = np.asarray(series, dtype=float)
    values = values - values.mean(axis=0, keepdims=True)
    n = len(values)
    n_fft = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(values, n=n_fft, axis=0)
    denominator = np.arange(n, 0, -1, dtype=float)
    correlation = np.empty((n, values.shape[1], values.shape[1]))
    for i in range(values.shape[1]):
        for j in range(values.shape[1]):
            raw = np.fft.irfft(
                spectrum[:, i] * spectrum[:, j].conj(), n=n_fft
            )[:n]
            correlation[:, i, j] = raw / denominator
    return correlation

def autocorrelation_scalar(series):
    values = np.asarray(series, dtype=float)
    values = values - values.mean()
    n = len(values)
    n_fft = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(values, n=n_fft)
    raw = np.fft.irfft(spectrum * spectrum.conj(), n=n_fft)[:n]
    return raw / np.arange(n, 0, -1, dtype=float)

p_corr = autocorrelation_matrix(polarization_series)
lag_fs = np.arange(len(p_corr)) * dt_fs
alpha = susceptibility_series
alpha_iso = np.trace(alpha, axis1=1, axis2=2) / 3.0
alpha_aniso_components = np.column_stack(
    [
        (alpha[:, 0, 0] - alpha[:, 1, 1]) / np.sqrt(2.0),
        (alpha[:, 1, 1] - alpha[:, 2, 2]) / np.sqrt(2.0),
        (alpha[:, 2, 2] - alpha[:, 0, 0]) / np.sqrt(2.0),
        np.sqrt(3.0) * alpha[:, 0, 1],
        np.sqrt(3.0) * alpha[:, 1, 2],
        np.sqrt(3.0) * alpha[:, 2, 0],
    ]
)
alpha_iso_corr = autocorrelation_scalar(alpha_iso)
alpha_aniso_corr = sum(
    autocorrelation_scalar(alpha_aniso_components[:, i])
    for i in range(alpha_aniso_components.shape[1])
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for axis in range(3):
    denom = max(float(p_corr[0, axis, axis]), 1.0e-30)
    axes[0].plot(lag_fs / 1000, p_corr[:, axis, axis] / denom,
                 label=f"component {axis}")
axes[0].set(xlabel="lag (ps)", ylabel="normalized ACF",
            title="Polarization autocorrelation")
axes[0].legend()
axes[1].plot(lag_fs / 1000,
             alpha_iso_corr / max(float(alpha_iso_corr[0]), 1.0e-30),
             label="isotropic susceptibility")
axes[1].plot(lag_fs / 1000,
             alpha_aniso_corr / max(float(alpha_aniso_corr[0]), 1.0e-30),
             label="anisotropic tensor")
axes[1].set(xlabel="lag (ps)", ylabel="normalized ACF",
            title="Polarizability autocorrelation")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Estimate the frequency-dependent dielectric tensor and spectra

The tensor fluctuation-dissipation estimate uses the one-sided transform
$\epsilon(\omega)=\epsilon_\infty+V/(\epsilon_0 k_BT)[C(0)-i\omega	ilde C(\omega)]$,
with the Fourier convention $e^{-i\omega t}$. The IR and Raman curves are relative classical spectra; the Raman proxy omits incident-laser, Bose-population, and absolute-intensity factors.


In [ ]:
n = len(time_fs)
n_fft = 1 << (2 * n - 1).bit_length()
window = 0.5 * (
    1.0 + np.cos(np.pi * lag_fs / (lag_fs[-1] + dt_fs))
)
p_corr_windowed = p_corr * window[:, None, None]
p_transform = np.fft.rfft(p_corr_windowed, n=n_fft, axis=0) * dt_fs

frequency_hz = np.fft.rfftfreq(n_fft, d=dt_fs * 1.0e-15)
frequency_cm = frequency_hz / (299792458.0 * 100.0)
omega_rad_fs = 2.0 * np.pi * np.fft.rfftfreq(n_fft, d=dt_fs)

elementary_charge = 1.602176634e-19
epsilon0_SI = 8.8541878128e-12
boltzmann = 1.380649e-23
polarization_unit_SI = elementary_charge / 1.0e-20
volume_m3 = volume_A3 * 1.0e-30
fluctuation_prefactor = (
    volume_m3 * polarization_unit_SI**2
    / (epsilon0_SI * boltzmann * temperature_K)
)
epsilon_infinity = np.eye(3) + np.mean(susceptibility_series, axis=0)
epsilon_omega = (
    epsilon_infinity[None, :, :]
    + fluctuation_prefactor
    * (
        p_corr[0][None, :, :]
        - 1j * omega_rad_fs[:, None, None] * p_transform
    )
)

alpha_iso_transform = np.fft.rfft(
    alpha_iso_corr * window, n=n_fft
) * dt_fs
alpha_aniso_transform = np.fft.rfft(
    alpha_aniso_corr * window, n=n_fft
) * dt_fs
ir_proxy = frequency_cm**2 * np.maximum(
    np.trace(p_transform.real, axis1=1, axis2=2), 0.0
)
raman_proxy = frequency_cm**2 * (
    45.0 * np.maximum(alpha_iso_transform.real, 0.0)
    + 7.0 * np.maximum(alpha_aniso_transform.real, 0.0)
)

show = frequency_cm <= 1600.0
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for i in range(3):
    axes[0, 0].plot(frequency_cm[show], epsilon_omega.real[show, i, i],
                    label=f"epsilon {i}{i}")
    axes[0, 0].plot(frequency_cm[show], -epsilon_omega.imag[show, i, i],
                    linestyle="--", label=f"loss {i}{i}")
axes[0, 0].set(xlabel="wavenumber (cm⁻¹)", ylabel="relative permittivity",
              title="Frequency-dependent dielectric tensor")
axes[0, 0].legend(fontsize=7)
axes[0, 1].plot(frequency_cm[show], ir_proxy[show], color="#2f6b8a")
axes[0, 1].set(xlabel="wavenumber (cm⁻¹)", ylabel="relative intensity",
               title="IR proxy from polarization fluctuations")
axes[1, 0].plot(frequency_cm[show], raman_proxy[show], color="#9b4f96")
axes[1, 0].set(xlabel="Raman shift (cm⁻¹)", ylabel="relative intensity",
               title="Raman proxy from susceptibility fluctuations")
axes[1, 1].plot(frequency_cm[show], np.trace(epsilon_omega.real[show], axis1=1, axis2=2) / 3,
                label="mean Re epsilon")
axes[1, 1].plot(frequency_cm[show], -np.trace(epsilon_omega.imag[show], axis1=1, axis2=2) / 3,
                label="mean dielectric loss")
axes[1, 1].set(xlabel="wavenumber (cm⁻¹)", ylabel="relative permittivity",
               title="Mean dielectric response")
axes[1, 1].legend()
for ax in axes.flat:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

print("static fluctuation estimate, diagonal:", np.diag(epsilon_omega[0].real))
print("epsilon infinity, diagonal:", np.diag(epsilon_infinity))
print("The frequency grid is zero-padded; its spacing does not improve the physical resolution of this short trajectory.")


## Convergence notes

The MD is fixed-cell, classical, and at zero field. Reliable vibrational spectra need longer trajectories, checks over independent initial velocities and blocks, timestep and thermostat sensitivity, and model validation over the sampled configurations. The dielectric estimate also depends on the periodic boundary convention and finite-size polarization fluctuations.

For methods context, see the [MACEField paper](https://doi.org/10.1038/S41467-025-59304-1) and the [frequency-dependent dielectric function tutorial in Calorine](https://calorine.materialsmodeling.org/get_started/dielectric_functions.html).
